# SQL Query Comparison Test

This notebook compares the SQL query from `seed_qna.csv` with the working SQL from `cohortmatching.sql` to verify they return identical results.


In [1]:
import csv
import os
import pandas as pd
from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import RealDictCursor

# Load environment variables
load_dotenv()


True

In [2]:
# Database connection configuration
# Update these with your actual database credentials
DATABASE_URL = os.getenv("DATABASE_URL")

# Or use individual parameters:
DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'amili-datalake-prod-new.cluster-cn7qhsjnw3bu.ap-southeast-1.rds.amazonaws.com'),
    'port': int(os.getenv('POSTGRES_PORT', 5432)),
    'database': os.getenv('POSTGRES_DB', 'app_db'),
    'user': os.getenv('POSTGRES_USER'),
    'password': os.getenv('POSTGRES_PASSWORD'),
}

print("Database configuration loaded")


Database configuration loaded


In [3]:
# Read SQL from seed_qna.csv
def read_sql_from_seed_csv():
    csv_path = "business/seed_qna.csv"
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        rows = list(reader)
        
        # Find the cohort matching query (row with ELEGANCE, AMD, EMULSION)
        for row in rows:
            if len(row) > 0 and 'ELEGANCE' in row[0] and 'AMD' in row[0] and 'EMULSION' in row[0] and 'build an unhealthy cohort' in row[0]:
                sql = row[1]
                print(f"✅ Found cohort matching query in seed_qna.csv")
                print(f"   SQL length: {len(sql)} characters")
                print(f"   First 100 chars: {sql[:100]}...")
                return sql
    
    raise ValueError("Could not find cohort matching query in seed_qna.csv")

seed_sql = read_sql_from_seed_csv()


✅ Found cohort matching query in seed_qna.csv
   SQL length: 19195 characters
   First 100 chars: WITH
---------------------------------------------------------------------
-- 1. ELEGANCE UNHEALTHY ...


In [4]:
# Read SQL from cohortmatching.sql file
def read_sql_from_file():
    sql_path = "/Users/vu.minh.hung/Downloads/cohortmatching.sql"
    with open(sql_path, 'r', encoding='utf-8') as f:
        sql = f.read()
        print(f"✅ Read SQL from cohortmatching.sql")
        print(f"   SQL length: {len(sql)} characters")
        print(f"   First 100 chars: {sql[:100]}...")
        return sql

file_sql = read_sql_from_file()


✅ Read SQL from cohortmatching.sql
   SQL length: 19195 characters
   First 100 chars: WITH
---------------------------------------------------------------------
-- 1. ELEGANCE UNHEALTHY ...


In [5]:
# Compare the SQL queries
print("=" * 80)
print("SQL COMPARISON")
print("=" * 80)
print(f"\nSeed SQL length: {len(seed_sql)}")
print(f"File SQL length: {len(file_sql)}")
print(f"Length difference: {abs(len(seed_sql) - len(file_sql))}")

# Check if they're identical
if seed_sql == file_sql:
    print("\n✅ SQL queries are IDENTICAL")
else:
    print("\n⚠️  SQL queries are DIFFERENT")
    # Find first difference
    min_len = min(len(seed_sql), len(file_sql))
    for i in range(min_len):
        if seed_sql[i] != file_sql[i]:
            print(f"\nFirst difference at position {i}:")
            print(f"  Seed: {repr(seed_sql[i])}")
            print(f"  File: {repr(file_sql[i])}")
            start = max(0, i - 50)
            end = min(len(seed_sql), i + 50)
            print(f"\n  Seed context: ...{seed_sql[start:end]}...")
            print(f"  File context: ...{file_sql[start:end]}...")
            break

# Check for table name quotes
print("\n" + "=" * 80)
print("CHECKING TABLE NAME QUOTES")
print("=" * 80)
table_name = '10_elegance_other_information'
seed_quoted = f'"{table_name}"' in seed_sql
file_quoted = f'"{table_name}"' in file_sql
print(f"\nTable '{table_name}':")
print(f"  In seed SQL - properly quoted: {seed_quoted}")
print(f"  In file SQL - properly quoted: {file_quoted}")


SQL COMPARISON

Seed SQL length: 19195
File SQL length: 19195
Length difference: 0

✅ SQL queries are IDENTICAL

CHECKING TABLE NAME QUOTES

Table '10_elegance_other_information':
  In seed SQL - properly quoted: True
  In file SQL - properly quoted: True


In [6]:
# Function to execute SQL and return DataFrame
def execute_sql(sql_query, query_name):
    """Execute SQL query and return results as DataFrame."""
    try:
        if DATABASE_URL:
            conn = psycopg2.connect(DATABASE_URL)
        else:
            conn = psycopg2.connect(**DB_CONFIG)
        
        cursor = conn.cursor(cursor_factory=RealDictCursor)
        
        print(f"\n🔍 Executing {query_name}...")
        cursor.execute(sql_query)
        
        rows = cursor.fetchall()
        df = pd.DataFrame(rows)
        
        cursor.close()
        conn.close()
        
        print(f"✅ Query executed successfully")
        print(f"   Rows returned: {len(df)}")
        print(f"   Columns: {list(df.columns)}")
        
        return df
        
    except Exception as e:
        print(f"❌ Error executing {query_name}: {str(e)}")
        raise


In [14]:
# Execute seed SQL query
print("=" * 80)
print("EXECUTING SEED SQL QUERY")
print("=" * 80)
df_seed = execute_sql(seed_sql, "Seed SQL (from seed_qna.csv)")
df_seed[:5]


EXECUTING SEED SQL QUERY

🔍 Executing Seed SQL (from seed_qna.csv)...
✅ Query executed successfully
   Rows returned: 562
   Columns: ['subject_id', 'gender', 'age_no', 'race', 'sequencing_id', 'hypertension_ind', 'systolic_blood_pressure_val', 'diastolic_blood_pressure_val', 'diabetes_ind', 'hba1c_val', 'obese_ind', 'bmi_val', 'hyperlipidaemia_ind', 'total_cholesterol_val', 'triglyceride_val', 'health_status', 'dataset_source']


,subject_id,gender,age_no,race,sequencing_id,hypertension_ind,systolic_blood_pressure_val,diastolic_blood_pressure_val,diabetes_ind,hba1c_val,obese_ind,bmi_val,hyperlipidaemia_ind,total_cholesterol_val,triglyceride_val,health_status,dataset_source
0,FLV118,Female,23,Chinese,B18FLV118,0,None,None,0,None,Obese,35.96,0,None,None,Unhealthy,EMULSION
1,AMD0808,Female,25,Chinese,B15AMD0808,0,None,None,0,None,Obese,29.74,0,None,None,Unhealthy,AMD
2,AMD1033,Female,26,Chinese,B046AMD1033,0,None,None,0,None,Obese,28.65,0,None,None,Unhealthy,AMD
3,FLV047,Female,26,Chinese,B18FLV047,0,None,None,0,None,Obese,34.73,0,None,None,Unhealthy,EMULSION
4,AMD1421,Female,27,Chinese,B080AMD1421,0,None,None,0,None,Obese,31.64,0,None,None,Unhealthy,AMD


In [8]:
# Execute file SQL query
print("=" * 80)
print("EXECUTING FILE SQL QUERY")
print("=" * 80)
df_file = execute_sql(file_sql, "File SQL (from cohortmatching.sql)")


EXECUTING FILE SQL QUERY

🔍 Executing File SQL (from cohortmatching.sql)...
✅ Query executed successfully
   Rows returned: 562
   Columns: ['subject_id', 'gender', 'age_no', 'race', 'sequencing_id', 'hypertension_ind', 'systolic_blood_pressure_val', 'diastolic_blood_pressure_val', 'diabetes_ind', 'hba1c_val', 'obese_ind', 'bmi_val', 'hyperlipidaemia_ind', 'total_cholesterol_val', 'triglyceride_val', 'health_status', 'dataset_source']


In [9]:
# Compare results
print("=" * 80)
print("COMPARING RESULTS")
print("=" * 80)

print(f"\nSeed query returned: {len(df_seed)} rows")
print(f"File query returned: {len(df_file)} rows")

if len(df_seed) == len(df_file):
    print("✅ Same number of rows")
else:
    print("❌ Different number of rows")

# Check if DataFrames are identical
if df_seed.equals(df_file):
    print("\n✅ Results are IDENTICAL")
else:
    print("\n⚠️  Results are DIFFERENT")
    
    # Compare column by column
    if set(df_seed.columns) != set(df_file.columns):
        print(f"\nColumn mismatch:")
        print(f"  Seed columns: {set(df_seed.columns)}")
        print(f"  File columns: {set(df_file.columns)}")
    else:
        print(f"\n✅ Same columns: {list(df_seed.columns)}")
        
        # Find differences
        if len(df_seed) == len(df_file):
            diff_mask = df_seed != df_file
            diff_rows = diff_mask.any(axis=1)
            if diff_rows.any():
                print(f"\n⚠️  Found {diff_rows.sum()} rows with differences")
                print("\nFirst few differing rows:")
                print(df_seed[diff_rows].head())
                print("\nvs")
                print(df_file[diff_rows].head())


COMPARING RESULTS

Seed query returned: 562 rows
File query returned: 562 rows
✅ Same number of rows

✅ Results are IDENTICAL


In [10]:
# Display first 5 records from seed query
print("=" * 80)
print("FIRST 5 RECORDS FROM SEED QUERY")
print("=" * 80)
print("\n")
display(df_seed.head(5))


FIRST 5 RECORDS FROM SEED QUERY




,subject_id,gender,age_no,race,sequencing_id,hypertension_ind,systolic_blood_pressure_val,diastolic_blood_pressure_val,diabetes_ind,hba1c_val,obese_ind,bmi_val,hyperlipidaemia_ind,total_cholesterol_val,triglyceride_val,health_status,dataset_source
0,FLV118,Female,23,Chinese,B18FLV118,0,None,None,0,None,Obese,35.96,0,None,None,Unhealthy,EMULSION
1,AMD0808,Female,25,Chinese,B15AMD0808,0,None,None,0,None,Obese,29.74,0,None,None,Unhealthy,AMD
2,AMD1033,Female,26,Chinese,B046AMD1033,0,None,None,0,None,Obese,28.65,0,None,None,Unhealthy,AMD
3,FLV047,Female,26,Chinese,B18FLV047,0,None,None,0,None,Obese,34.73,0,None,None,Unhealthy,EMULSION
4,AMD1421,Female,27,Chinese,B080AMD1421,0,None,None,0,None,Obese,31.64,0,None,None,Unhealthy,AMD


In [11]:
# Display first 5 records from file query
print("=" * 80)
print("FIRST 5 RECORDS FROM FILE QUERY")
print("=" * 80)
print("\n")
display(df_file.head(5))


FIRST 5 RECORDS FROM FILE QUERY




,subject_id,gender,age_no,race,sequencing_id,hypertension_ind,systolic_blood_pressure_val,diastolic_blood_pressure_val,diabetes_ind,hba1c_val,obese_ind,bmi_val,hyperlipidaemia_ind,total_cholesterol_val,triglyceride_val,health_status,dataset_source
0,FLV118,Female,23,Chinese,B18FLV118,0,None,None,0,None,Obese,35.96,0,None,None,Unhealthy,EMULSION
1,AMD0808,Female,25,Chinese,B15AMD0808,0,None,None,0,None,Obese,29.74,0,None,None,Unhealthy,AMD
2,AMD1033,Female,26,Chinese,B046AMD1033,0,None,None,0,None,Obese,28.65,0,None,None,Unhealthy,AMD
3,FLV047,Female,26,Chinese,B18FLV047,0,None,None,0,None,Obese,34.73,0,None,None,Unhealthy,EMULSION
4,AMD1421,Female,27,Chinese,B080AMD1421,0,None,None,0,None,Obese,31.64,0,None,None,Unhealthy,AMD


In [12]:
# Verify expected records
print("=" * 80)
print("VERIFYING EXPECTED RECORDS")
print("=" * 80)

expected_records = [
    {"subject_id": "FLV118", "gender": "Female", "age_no": 23, "race": "Chinese", "sequencing_id": "B18FLV118", "bmi_val": 35.96, "health_status": "Unhealthy", "dataset_source": "EMULSION"},
    {"subject_id": "AMD0808", "gender": "Female", "age_no": 25, "race": "Chinese", "sequencing_id": "B15AMD0808", "bmi_val": 29.74, "health_status": "Unhealthy", "dataset_source": "AMD"},
    {"subject_id": "AMD1033", "gender": "Female", "age_no": 26, "race": "Chinese", "sequencing_id": "B046AMD1033", "bmi_val": 28.65, "health_status": "Unhealthy", "dataset_source": "AMD"},
    {"subject_id": "FLV047", "gender": "Female", "age_no": 26, "race": "Chinese", "sequencing_id": "B18FLV047", "bmi_val": 34.73, "health_status": "Unhealthy", "dataset_source": "EMULSION"},
    {"subject_id": "AMD1421", "gender": "Female", "age_no": 27, "race": "Chinese", "sequencing_id": "B080AMD1421", "bmi_val": 31.64, "health_status": "Unhealthy", "dataset_source": "AMD"},
]

print("\nChecking if expected records are in results...")

for i, expected in enumerate(expected_records, 1):
    subject_id = expected["subject_id"]
    
    # Check in seed results
    seed_match = df_seed[df_seed["subject_id"] == subject_id]
    
    if len(seed_match) > 0:
        row = seed_match.iloc[0]
        matches = (
            str(row["gender"]) == expected["gender"] and
            int(row["age_no"]) == expected["age_no"] and
            str(row["race"]) == expected["race"] and
            str(row["sequencing_id"]) == expected["sequencing_id"] and
            abs(float(row["bmi_val"]) - expected["bmi_val"]) < 0.01 and
            str(row["health_status"]) == expected["health_status"] and
            str(row["dataset_source"]) == expected["dataset_source"]
        )
        
        if matches:
            print(f"✅ Record {i} ({subject_id}): Found and matches expected values")
        else:
            print(f"⚠️  Record {i} ({subject_id}): Found but values differ")
            print(f"   Expected: {expected}")
            print(f"   Got: subject_id={row['subject_id']}, gender={row['gender']}, age_no={row['age_no']}, race={row['race']}, sequencing_id={row['sequencing_id']}, bmi_val={row['bmi_val']}, health_status={row['health_status']}, dataset_source={row['dataset_source']}")
    else:
        print(f"❌ Record {i} ({subject_id}): NOT FOUND in results")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\nTotal rows returned by seed query: {len(df_seed)}")
print(f"Total rows returned by file query: {len(df_file)}")
print(f"\nQueries return same results: {df_seed.equals(df_file) if len(df_seed) == len(df_file) else False}")


VERIFYING EXPECTED RECORDS

Checking if expected records are in results...
✅ Record 1 (FLV118): Found and matches expected values
✅ Record 2 (AMD0808): Found and matches expected values
✅ Record 3 (AMD1033): Found and matches expected values
✅ Record 4 (FLV047): Found and matches expected values
✅ Record 5 (AMD1421): Found and matches expected values

SUMMARY

Total rows returned by seed query: 562
Total rows returned by file query: 562

Queries return same results: True


In [13]:
# Show full results (if needed)
print("=" * 80)
print("FULL RESULTS FROM SEED QUERY")
print("=" * 80)
display(df_seed)


FULL RESULTS FROM SEED QUERY


,subject_id,gender,age_no,race,sequencing_id,hypertension_ind,systolic_blood_pressure_val,diastolic_blood_pressure_val,diabetes_ind,hba1c_val,obese_ind,bmi_val,hyperlipidaemia_ind,total_cholesterol_val,triglyceride_val,health_status,dataset_source
0,FLV118,Female,23,Chinese,B18FLV118,0,None,None,0,None,Obese,35.96,0,None,None,Unhealthy,EMULSION
1,AMD0808,Female,25,Chinese,B15AMD0808,0,None,None,0,None,Obese,29.74,0,None,None,Unhealthy,AMD
2,AMD1033,Female,26,Chinese,B046AMD1033,0,None,None,0,None,Obese,28.65,0,None,None,Unhealthy,AMD
3,FLV047,Female,26,Chinese,B18FLV047,0,None,None,0,None,Obese,34.73,0,None,None,Unhealthy,EMULSION
4,AMD1421,Female,27,Chinese,B080AMD1421,0,None,None,0,None,Obese,31.64,0,None,None,Unhealthy,AMD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557,AMD0990,Male,21,Others,B010AMD0990,0,None,None,0,None,Normal,20.75,0,None,None,Healthy,AMD_CONTROL
558,AMD0735,Male,22,Others,B13AMD0735,0,None,None,0,None,Normal,21.38,0,None,None,Healthy,AMD_CONTROL
559,AMD0600,Male,24,Others,B13AMD0600,0,None,None,0,None,Normal,22.63,0,None,None,Healthy,AMD_CONTROL
560,AMD1393,Male,30,Others,B079AMD1393,0,None,None,0,None,Normal,22.28,0,None,None,Healthy,AMD_CONTROL
